In [ ]:
from pyspark.sql import DataFrame, functions as F

from transforms.calibration import (
    to_long_format,
    fit_all_calibration_curves,
    calibrate_matches,
)
from transforms.elo import build_elo
from transforms.scoring import build_coupon_predictions

SILVER_TABLE = "stryktipset.silver.matches"
PREDICTIONS_TABLE = "stryktipset.gold.coupon_predictions"

In [ ]:
def load_matches(table: str = SILVER_TABLE) -> DataFrame:
    """Silver, unscoped -- every league and country, played and not."""
    return spark.table(table)


def open_draw_number(matches: DataFrame) -> int:
    """The newest draw in silver.

    01 fetches the open draw now, so this is normally the coupon that has
    not been played yet. Once its matches finish it is simply the most
    recent completed one, and scoring it again is harmless.
    """
    return matches.agg(F.max("draw_number")).first()[0]

In [ ]:
def write_predictions(df: DataFrame, table: str = PREDICTIONS_TABLE) -> None:
    """Create the table on first run, then MERGE on match_id.

    A coupon is scored repeatedly through the week -- streck keeps moving
    until the draw closes -- so re-running must update a row in place
    rather than append a duplicate. The wildcard UPDATE SET * / INSERT *
    forms are used because those are the ones Delta's schema evolution
    actually supports.
    """
    if not spark.catalog.tableExists(table):
        df.write.format("delta").saveAsTable(table)
        print(f"created {table}")
        return

    source_view = "coupon_source"
    df.createOrReplaceTempView(source_view)
    spark.sql(f"""
        MERGE WITH SCHEMA EVOLUTION INTO {table} AS target
        USING {source_view} AS source
        ON target.match_id = source.match_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

In [ ]:
def show(rows) -> None:
    """Print the coupon the way you would read it on a Thursday."""
    header = f"{'#':<3}{'match':<36}{'league':<22}{'streck 1/X/2':<20}{'elo 1':>6}"
    print(header)
    print("-" * len(header))
    for row in rows:
        fixture = f"{row['home_team']} - {row['away_team']}"
        league = row["league"] or ""
        if row["streck_1"] is None:
            streck = "-"
        else:
            streck = "{:.0%}/{:.0%}/{:.0%}".format(
                row["streck_1"], row["streck_x"], row["streck_2"]
            )
        elo = "-" if row["elo_prob_1"] is None else "{:.0%}".format(row["elo_prob_1"])
        print(f"{row['event_number']:<3}{fixture[:35]:<36}{league[:21]:<22}{streck:<20}{elo:>6}")

In [ ]:
def main():
    """Score the open coupon with everything the pipeline knows.

    Calibration is refitted here from silver's history rather than loading
    05's logged MLflow model. That keeps this notebook deterministic and
    free of a model-registry lookup; swapping to the registered model is a
    refinement, not a requirement.
    """
    matches = load_matches()

    models = fit_all_calibration_curves(to_long_format(matches))
    calibrated = calibrate_matches(matches, models)

    elo = build_elo(matches)

    draw_number = open_draw_number(matches)
    predictions = build_coupon_predictions(calibrated, elo, draw_number)

    rows = predictions.collect()
    print(f"draw {draw_number} -- {len(rows)} matches")
    print()
    show(rows)

    missing = [row["event_number"] for row in rows if row["elo_home"] is None]
    if missing:
        print()
        print(f"no Elo for event(s) {missing}: a team with no prior match in the data")

    write_predictions(predictions)
    print()
    print(f"written to {PREDICTIONS_TABLE}")

    dbutils.notebook.exit(
        f"draw={draw_number} matches={len(rows)} without_elo={len(missing)}"
    )

main()